## Welcome to PyJAGS

**PyJAGS** is a Python interface to [JAGS](https://mcmc-jags.sourceforge.io/) (Just Another Gibbs Sampler), a program for analyzing Bayesian statistical models using Markov Chain Monte Carlo (MCMC) simulation. PyJAGS lets you define models, compile them, and draw samples entirely from Python.

**JAGS** is a mature, widely-used tool for Bayesian inference. You describe your statistical model using the BUGS language, and JAGS automatically constructs an efficient MCMC sampler to draw from the posterior distribution.

**Bayesian inference** is a framework for updating your beliefs about unknown quantities (parameters) after observing data. Instead of producing a single "best" estimate, Bayesian methods give you a full probability distribution over the parameters, which naturally quantifies uncertainty.

### What you'll learn in this notebook

- How to write a simple JAGS model in Python
- How to validate model syntax before compiling
- How to compile a model, draw MCMC samples, and inspect results
- How to use ArviZ for summary statistics and diagnostic plots
- How to use seeds for reproducible sampling
- How to load models from files

## Setup

In [ ]:
import pyjags
import numpy as np
import arviz as az
import matplotlib.pyplot as plt

pyjags.version_info()

## Your First Model: Estimating a Normal Mean

Suppose we have collected 30 measurements from some process, and we believe the data are roughly normally distributed. We want to estimate the **mean** and **standard deviation** of the underlying distribution.

Let's start by generating some synthetic data so we know the true values (mean = 5.0, standard deviation = 2.0) and can verify that our model recovers them.

In [ ]:
np.random.seed(42)
y = np.random.normal(loc=5.0, scale=2.0, size=30)
print(f"Sample mean:  {y.mean():.2f}")
print(f"Sample std:   {y.std(ddof=1):.2f}")

Now we write the JAGS model. The BUGS language used by JAGS is declarative: you specify the probabilistic relationships, and JAGS figures out how to sample from the posterior.

A few things to note:
- **Priors**: We place vague (weakly informative) priors on `mu` and `sigma` so the data can speak for themselves.
- **Precision**: JAGS parameterizes the normal distribution with *precision* (1/variance) rather than variance or standard deviation. We define `tau` as a derived quantity.
- **Likelihood**: The `for` loop says that each observation `y[i]` is drawn from a normal distribution with mean `mu` and precision `tau`.

In [ ]:
model_code = """
model {
    # Priors
    mu ~ dnorm(0, 0.001)        # vague normal prior on the mean
    sigma ~ dunif(0, 100)       # uniform prior on std deviation
    tau <- pow(sigma, -2)       # JAGS uses precision, not variance

    # Likelihood
    for (i in 1:N) {
        y[i] ~ dnorm(mu, tau)
    }
}
"""

## Validating the Model

Before compiling and running, it's a good idea to check that the model syntax is valid. The `check_model()` function parses the model code and raises a clear error if there are any problems.

In [ ]:
pyjags.check_model(code=model_code)

Here's what happens when the model contains a syntax error. The error message includes annotated source context showing exactly where the problem is.

In [ ]:
bad_model_code = """
model {
    mu ~ dnorm(0, 0.001)
    sigma ~ dunif(0, 100
}
"""

try:
    pyjags.check_model(code=bad_model_code)
except Exception as e:
    print(f"Error caught: {e}")

## Compiling and Sampling

Now we compile the model by creating a `Model` object. We pass:
- `code`: the JAGS model string
- `data`: a dictionary mapping variable names to their observed values
- `chains`: the number of independent MCMC chains (4 is a common choice for convergence diagnostics)
- `adapt`: the number of adaptation steps (JAGS tunes its samplers during this phase)
- `seed`: an integer seed for reproducible results (uses `numpy.random.SeedSequence` to derive independent per-chain seeds)

In [ ]:
data = dict(y=y, N=len(y))

model = pyjags.Model(
    code=model_code,
    data=data,
    chains=4,
    adapt=1000,
    progress_bar=False,
    seed=42,
)

The model's `repr` gives a quick overview of its state:

In [ ]:
model

## Inspecting the Model

Before sampling, we can inspect some properties of the compiled model.

In [ ]:
print(f"Adapted:           {model.is_adapted}")
print(f"Current iteration: {model.iteration}")
print(f"Samplers:          {model.samplers}")

## Drawing Samples

MCMC sampling is done in two phases:

1. **Burn-in**: We run the sampler for some iterations and discard the results. This lets the chains move away from their starting values and settle into the region of high posterior probability.
2. **Production**: We collect the samples we'll actually use for inference.

To discard burn-in, we call `sample()` with `vars=[]` (monitor no variables).

In [ ]:
model.sample(1000, vars=[])  # discard burn-in

Now we draw 5000 production samples of `mu` and `sigma`.

In [ ]:
samples = model.sample(5000, vars=["mu", "sigma"])

The returned dictionary maps variable names to numpy arrays. The shape convention in PyJAGS is `(*variable_dims, iterations, chains)`. For scalar parameters like `mu` and `sigma`, the variable dimension is 1.

In [ ]:
for name, arr in samples.items():
    print(f"{name}: shape {arr.shape}")
# mu: shape (1, 5000, 4) -> (variable_dims, iterations, chains)

## Analyzing Results with ArviZ

[ArviZ](https://python.arviz.org/) is the standard Python library for exploratory analysis of Bayesian models. PyJAGS provides a `from_pyjags()` function that converts sample dictionaries directly into ArviZ inference data objects.

You can optionally attach your observed data and constants for a self-contained analysis object.

In [ ]:
idata = pyjags.from_pyjags(
    samples,
    observed_data={"y": y},
    constant_data={"N": np.array(len(y))},
)

The returned object carries metadata about the inference library and version:

In [ ]:
print(idata.attrs)

### Summary Statistics

The `az.summary()` function computes posterior means, standard deviations, credible intervals, effective sample sizes (ESS), and the R-hat convergence diagnostic. R-hat values close to 1.0 indicate good convergence.

In [ ]:
az.summary(idata, var_names=["mu", "sigma"])

### Trace Plot

Trace plots show the sampled values over time (left) and the posterior distribution (right) for each parameter. Well-mixed chains look like "fuzzy caterpillars" in the trace plot.

In [ ]:
az.plot_trace(idata, var_names=["mu", "sigma"])
plt.tight_layout()
plt.show()

### Forest Plot

Forest plots summarize the posterior distributions with credible intervals, making it easy to compare parameters at a glance.

In [ ]:
az.plot_forest(idata, var_names=["mu", "sigma"])
plt.show()

## Reproducibility with Seeds

Scientific analyses should be reproducible. The `seed` parameter in `Model()` uses `numpy.random.SeedSequence` to derive statistically independent per-chain seeds. Running the same code with the same seed will produce identical results.

In [ ]:
m1 = pyjags.Model(code=model_code, data=data, chains=4, adapt=1000, progress_bar=False, seed=42)
m1.sample(1000, vars=[])
s1 = m1.sample(1000, vars=["mu"])

m2 = pyjags.Model(code=model_code, data=data, chains=4, adapt=1000, progress_bar=False, seed=42)
m2.sample(1000, vars=[])
s2 = m2.sample(1000, vars=["mu"])

print(f"Results are identical: {np.array_equal(s1['mu'], s2['mu'])}")

## Loading Models from Files

For larger models, it's convenient to store the JAGS code in a separate `.bug` file. PyJAGS accepts `pathlib.Path` objects as well as plain strings for file paths.

In [ ]:
from pathlib import Path
import tempfile

model_file = Path(tempfile.mktemp(suffix=".bug"))
model_file.write_text(model_code)

model_from_file = pyjags.Model(
    file=model_file,
    data=data,
    chains=4,
    adapt=1000,
    progress_bar=False,
)
print(model_from_file)

# Clean up
model_file.unlink()

## Next Steps

Now that you've seen the basics, here are some directions to explore:

- **Eight Schools** notebook: A classic example of hierarchical (multi-level) modeling
- **Logistic Regression** notebook: Binary classification with Bayesian logistic regression
- **Trading Cost Estimation** notebook: A real-world financial application
- **Advanced Functionality** notebook: Covers iterative sampling, convergence criteria, HDF5 persistence, and more

### External resources

- [JAGS User Manual](https://mcmc-jags.sourceforge.io/) -- full reference for the BUGS modeling language and JAGS distributions
- [ArviZ Documentation](https://python.arviz.org/) -- comprehensive guide to Bayesian analysis and visualization in Python